# M2 — VERSION B: RR_FEATURES (QC-aware variant)

**Important:** Run Version B only after Version A has finished, its outputs have been saved, and its results have been reviewed. This notebook writes to a separate output directory and does not overwrite Version A.

## How to run

1. Keep `RUN_MODEL_TRAINING = False` in Cell 1.
2. Attach the same Kaggle inputs used by Version A: corrected dual-lead dataset, M1 reconstructed beat table, original annotation files, saved M0 row/person fold files, split fingerprint, and M0 reference OOF predictions.
3. Run all cells from top to bottom. Cells 1–7 load and validate data, construct Version B RR features, and build the two feature matrices. **No model is trained while the switch is False.**
4. Review the feature and QC outputs. Do not proceed if any validation fails.
5. Set `RUN_MODEL_TRAINING = True` in Cell 1 and use **Save Version → Save & Run All (Commit)**. Kaggle will execute the notebook in a fresh session, so all required datasets must remain attached as inputs.
6. Download the contents of `/kaggle/working/ECG_Project/05_M2_RR_FEATURES_VERSION_B/` when the run finishes.

## Version B RR policy (fixed before Version B model results)

- Build intervals from the complete original heartbeat-annotation sequence (105,078 annotations); the 31 boundary-only beats contribute to timing but are not model rows.
- Preserve all 105,047 M1 model rows and attach features by `(Record_ID, Sample_Index)`.
- Mask intervals containing `~`, `|`, `!`, `[` or `]` strictly between their endpoint beats.
- Additionally mask exactly `232_629048_629972` (Candidate 4, sample 629573). Do not insert a beat or split this interval.
- Keep `x`, `+`, and `"` context-only intervals unless a separate mask trigger is also present.
- Retain the two post-IVR E-to-E intervals as annotation-derived RR values and mark them explicitly in the QC sidecar.
- Do not mask by duration and do not clip long intervals.
- `mean_previous_10_RR` requires **all 10 immediately preceding interval positions to be valid**. If any is missing/masked, the mean is missing.
- First `pre_RR` and last `post_RR` are missing. Missing features are never zero-filled or globally imputed; use the same scikit-learn native-NaN Random Forest behavior as Version A.

**Only the interval QC policy and the strict 10-of-10 history rule differ from Version A.** Keep the saved M0 splits, labels, reference model, Random Forest parameters, and evaluation protocol unchanged.


In [1]:
#cell1
from pathlib import Path
import hashlib
import json
import platform
import sys
import time

import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support
)

# Keep False while checking the data and RR features (cells up to feature construction).
# Set True only for the unattended run: Save Version -> Save & Run All.
RUN_MODEL_TRAINING = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/ECG_Project/05_M2_RR_FEATURES_VERSION_B")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = ["N", "S", "V", "F", "Q"]
FOUR_CLASS_ORDER = ["N", "S", "V", "Q"]

EXPECTED_COUNTS = {
    "N": 90341,
    "S": 2781,
    "V": 7229,
    "F": 802,
    "Q": 3894
}

EXPECTED_BEATS = 105047
EXPECTED_FULL_ANNOTATION_BEATS = 105078
EXPECTED_BOUNDARY_EXCLUSIONS = 31
EXPECTED_RECORDS = 46
EXPECTED_PEOPLE = 45

SAMPLE_RATE_HZ = 360.0

N_SPLITS = 5
N_REPEATS = 5
SEED = 42

# Same settings as REFERENCE_MODEL.
TREES = 200
MAX_FEATURES = "log2"

BOOTSTRAP_SAMPLES = 1000
BOOTSTRAP_SEED = 20261105

EXPECTED_SPLIT_FINGERPRINT = (
    "5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9"
)

SIGNAL_COLS = (
    [f"MLII_{i}" for i in range(201)]
    + [f"Lead2_{i}" for i in range(201)]
)

AAMI_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V",
    "F": "F",
    "/": "Q", "f": "Q", "Q": "Q"
}

RR_FEATURES = [
    "pre_RR",
    "post_RR",
    "mean_previous_10_RR",
    "pre_RR_div_mean_previous_10_RR",
    "post_RR_div_pre_RR"
]

PROB_COLS = [f"Prob_{c}" for c in CLASS_ORDER]


INPUT_ROOT = Path("/kaggle/input")

def find_one(filename_options):
    matches = []

    for filename in filename_options:
        matches.extend(
            p for p in INPUT_ROOT.rglob(filename)
            if p.is_file()
        )

    # Remove duplicate paths
    matches = sorted(set(matches))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one matching file from "
            f"{filename_options}. Found {len(matches)}:\n"
            + "\n".join(str(p) for p in matches)
        )

    print("FOUND:", matches[0])
    return matches[0]



# Version B RR policy, frozen before Version B model evaluation.
#
# Timing source: the FULL original heartbeat annotation sequence per record
# (105,078 beats, including 31 boundary-only annotations). Features are attached
# only to the 105,047 M1/model rows by (Record_ID, Sample_Index).
#
# Mask an interval if it strictly contains ~, |, !, [ or ].
# Also mask exactly Record 232 interval 629048 -> 629972 (Candidate 4, sample 629573).
# Do not insert a new beat and do not split Candidate 4's interval.
# Keep x, + and \" as context-only markers unless a mask marker is also present.
# The two post-IVR E-to-E intervals remain numeric and carry a context flag.
# There is NO duration threshold and NO clipping.
#
# First beat pre_RR and last beat post_RR are missing.
# mean_previous_10_RR requires all ten immediately preceding interval positions
# to be finite/valid. A single missing/masked interval makes the mean missing.
# Ratios require finite inputs and a positive denominator.
# No zero filling or global imputation; RF must support NaN natively.

RR_WINDOW = 10
RR_MIN_VALID = 10
MASK_SYMBOLS = ("~", "|", "!", "[", "]")
KEEP_SYMBOLS = ("x", "+", '"')
BEAT_SYMBOLS = tuple(AAMI_MAP.keys())

# Version B's one explicit interval-level QC decision from the two-lead review.
CANDIDATE4_INTERVAL = (232, 629048, 629972)
CANDIDATE4_SAMPLE = 629573

# These are true E-to-E post-IVR intervals. Retain their measured RR values but
# keep their context explicit in the interval-level QC sidecar.
KNOWN_POST_IVR_INTERVALS = {
    (207, 590585, 591405),
    (207, 591405, 592308)
}

# Reconciliation against the reviewed Version A run and the audit decision.
EXPECTED_VERSION_A_MARKER_MASKED_INTERVALS = 687
EXPECTED_VERSION_B_MASKED_INTERVALS = 688

# Optional: CSV with columns Record_ID, Sample_Index, Symbol holding ALL annotation
# rows (beats and non-beat markers). If None, original annotation text files are
# searched under /kaggle/input.
ANNOTATION_TABLE_PATH = None

RR_POLICY = {
    "version": "B",
    "interval_source": "full original heartbeat annotation sequence per record",
    "mask_symbols": list(MASK_SYMBOLS),
    "keep_context_symbols": list(KEEP_SYMBOLS),
    "additional_interval_mask": {
        "record_id": CANDIDATE4_INTERVAL[0],
        "start_sample": CANDIDATE4_INTERVAL[1],
        "end_sample": CANDIDATE4_INTERVAL[2],
        "candidate_sample_not_inserted": CANDIDATE4_SAMPLE,
        "reason": "UNRESOLVED_TWO_LEAD_WAVEFORM_CANDIDATE_4"
    },
    "post_ivr_e_to_e_intervals": [list(x) for x in sorted(KNOWN_POST_IVR_INTERVALS)],
    "post_ivr_policy": "retain numeric RR and flag context unless a mask marker is inside",
    "clipping": "none",
    "duration_threshold": "none",
    "first_beat_pre_RR": "missing",
    "last_beat_post_RR": "missing",
    "mean_previous_10_RR": "all 10 immediately preceding interval positions must be valid; otherwise missing",
    "ratios": "missing unless all inputs are finite and denominator is positive",
    "imputation": "none; native NaN support in RandomForestClassifier",
    "expected_marker_masked_intervals": EXPECTED_VERSION_A_MARKER_MASKED_INTERVALS,
    "expected_total_masked_intervals": EXPECTED_VERSION_B_MASKED_INTERVALS
}

print("PHASE M2 — RR FEATURES — VERSION B")
print("Training enabled:", RUN_MODEL_TRAINING)
print("Output folder:", OUTPUT_DIR)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

print("Random Forest:", TREES, "trees")
print("Max features:", MAX_FEATURES)
print("Random seed:", SEED)

PHASE M2 — RR FEATURES — VERSION B
Training enabled: True
Output folder: /kaggle/working/ECG_Project/05_M2_RR_FEATURES_VERSION_B
Python: 3.13.18
NumPy: 2.1.3
Pandas: 2.3.3
Scikit-learn: 1.6.1
Random Forest: 200 trees
Max features: log2
Random seed: 42


In [2]:
#cell2
corrected_path = find_one([
    "MITBIH_MLII_DualLead_Dataset_corrected.csv"
])

beat_table_path = find_one([
    "validated_reconstructed_beat_table.csv.gz",
    "validated_reconstructed_beat_table.csv"
])

print("Corrected dataset:", corrected_path)
print("M1 beat table:", beat_table_path)


# Load corrected dataset.

data = pd.read_csv(
    corrected_path,
    low_memory=False
)

data.columns = [
    str(c).strip().strip('"').strip("'").strip()
    for c in data.columns
]

if data.shape != (106870, 404):
    raise ValueError(
        f"Unexpected dataset shape: {data.shape}"
    )

if any(c not in data.columns for c in SIGNAL_COLS):
    raise ValueError("Missing one or more waveform columns.")

data["Record_ID"] = pd.to_numeric(
    data["Record_ID"],
    errors="raise"
).astype(int)

data["Label"] = data["Label"].astype(str).str.strip()

data["AAMI_Label"] = data["Label"].map(AAMI_MAP)


# Keep the five AAMI classes.

aami_df = data[
    data["AAMI_Label"].isin(CLASS_ORDER)
].copy().reset_index(drop=True)


# Person mapping.

aami_df["Person_ID"] = [
    "person_201_202"
    if rid in (201, 202)
    else f"person_{rid}"
    for rid in aami_df["Record_ID"].to_numpy()
]


# M0 confirmed Row_Index was created after AAMI filtering.

aami_df["Row_Index"] = np.arange(
    len(aami_df),
    dtype=np.int64
)


# Validate counts.

counts = (
    aami_df["AAMI_Label"]
    .value_counts()
    .reindex(CLASS_ORDER, fill_value=0)
    .to_dict()
)

assert len(aami_df) == EXPECTED_BEATS
assert aami_df["Record_ID"].nunique() == EXPECTED_RECORDS
assert aami_df["Person_ID"].nunique() == EXPECTED_PEOPLE
assert counts == EXPECTED_COUNTS, counts

assert np.isfinite(
    aami_df[SIGNAL_COLS].to_numpy(dtype=np.float32)
).all()


# Load M1 reconstructed beat table.

beat_table = pd.read_csv(
    beat_table_path,
    compression="infer",
    low_memory=False
)

beat_table.columns = [
    str(c).strip().strip('"').strip("'").strip()
    for c in beat_table.columns
]

required = [
    "Record_ID",
    "Person_ID",
    "Sample_Index",
    "Original_Symbol",
    "AAMI_Label"
] + SIGNAL_COLS

missing = [c for c in required if c not in beat_table.columns]

if missing:
    raise ValueError(
        f"M1 beat table is missing columns: {missing[:15]}"
    )


beat_table["Record_ID"] = pd.to_numeric(
    beat_table["Record_ID"],
    errors="raise"
).astype(int)

beat_table["Sample_Index"] = pd.to_numeric(
    beat_table["Sample_Index"],
    errors="raise"
).astype(np.int64)

beat_table["Person_ID"] = (
    beat_table["Person_ID"].astype(str).str.strip()
)

beat_table["AAMI_Label"] = (
    beat_table["AAMI_Label"].astype(str).str.strip()
)

beat_table["Original_Symbol"] = (
    beat_table["Original_Symbol"].astype(str).str.strip()
)


# Verification gates.

if len(beat_table) != EXPECTED_BEATS:
    raise ValueError("M1 beat table has an unexpected number of rows.")

if beat_table[
    ["Record_ID", "Sample_Index"]
].duplicated().any():
    raise ValueError("Duplicate record/sample keys found.")

if not np.array_equal(
    beat_table["Record_ID"].to_numpy(),
    aami_df["Record_ID"].to_numpy()
):
    raise ValueError("Record sequence mismatch. STOP.")

if not np.array_equal(
    beat_table["AAMI_Label"].to_numpy(),
    aami_df["AAMI_Label"].to_numpy()
):
    raise ValueError("AAMI labels do not match row-by-row. STOP.")

if not np.array_equal(
    beat_table["Person_ID"].to_numpy(),
    aami_df["Person_ID"].to_numpy()
):
    raise ValueError("Person IDs do not match row-by-row. STOP.")


# Verify all 402 waveform values.

processed = aami_df[SIGNAL_COLS].to_numpy(dtype=np.float32)

reconstructed = beat_table[SIGNAL_COLS].to_numpy(dtype=np.float32)

if not np.array_equal(processed, reconstructed):

    mismatch = np.argwhere(processed != reconstructed)

    raise ValueError(
        "Waveform mismatch. Do not align by row order. "
        f"First mismatch: {mismatch[0].tolist()}"
    )


# Transfer verified beat positions.

aami_df["Sample_Index"] = (
    beat_table["Sample_Index"].to_numpy(dtype=np.int64)
)

aami_df["Original_Symbol"] = (
    beat_table["Original_Symbol"].to_numpy()
)


print("\nCORRECTED DATA AND M1 POSITIONS — PASS")
print("Valid beats:", len(aami_df))
print("Records:", aami_df.Record_ID.nunique())
print("People:", aami_df.Person_ID.nunique())
print("Class counts:", counts)
print("402 waveform values match row-by-row: PASS")
print("Sample positions loaded: PASS")

FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/MITBIH_MLII_DualLead_Dataset_corrected.csv
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/validated_reconstructed_beat_table.csv
Corrected dataset: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/MITBIH_MLII_DualLead_Dataset_corrected.csv
M1 beat table: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/validated_reconstructed_beat_table.csv

CORRECTED DATA AND M1 POSITIONS — PASS
Valid beats: 105047
Records: 46
People: 45
Class counts: {'N': 90341, 'S': 2781, 'V': 7229, 'F': 802, 'Q': 3894}
402 waveform values match row-by-row: PASS
Sample positions loaded: PASS


In [3]:
#cell3
row_path = find_one([
    "m0_row_fold_membership.csv.gz",
    "m0_row_fold_membership.csv"
])

person_path = find_one([
    "m0_person_fold_membership.csv"
])

fingerprint_path = find_one([
    "m0_split_fingerprint.txt"
])


row_fold = pd.read_csv(
    row_path,
    compression="infer"
)

person_fold = pd.read_csv(person_path)


for col in ["Row_Index", "Repeat", "Fold"]:

    row_fold[col] = pd.to_numeric(
        row_fold[col],
        errors="raise"
    ).astype(int)


for col in ["Repeat", "Fold"]:

    person_fold[col] = pd.to_numeric(
        person_fold[col],
        errors="raise"
    ).astype(int)


person_fold["Person_ID"] = (
    person_fold["Person_ID"].astype(str).str.strip()
)


# Verify fingerprint.

computed_fp = hashlib.sha256(
    person_fold
    .sort_values(["Repeat", "Person_ID"])
    .to_csv(
        index=False,
        lineterminator="\n"
    )
    .encode("utf-8")
).hexdigest()

saved_fp = fingerprint_path.read_text().strip()

assert saved_fp == EXPECTED_SPLIT_FINGERPRINT
assert computed_fp == EXPECTED_SPLIT_FINGERPRINT

assert len(row_fold) == EXPECTED_BEATS * N_REPEATS
assert len(person_fold) == EXPECTED_PEOPLE * N_REPEATS

assert not row_fold.duplicated(
    ["Repeat", "Row_Index"]
).any()

assert not person_fold.duplicated(
    ["Repeat", "Person_ID"]
).any()

assert set(row_fold.Row_Index) == set(aami_df.Row_Index)
assert set(person_fold.Person_ID) == set(aami_df.Person_ID)


# Confirm row-level and person-level fold assignments agree.

row_to_person = aami_df.set_index(
    "Row_Index"
)["Person_ID"]

row_fold["Person_ID"] = row_fold["Row_Index"].map(
    row_to_person
)

if row_fold["Person_ID"].isna().any():
    raise ValueError("Some Row_Index values have no person mapping.")

joined = row_fold.merge(
    person_fold,
    on=["Repeat", "Person_ID"],
    suffixes=("_row", "_person"),
    how="left",
    validate="many_to_one"
)

if joined["Fold_person"].isna().any():
    raise ValueError("Missing person-level fold assignments.")

if not (
    joined["Fold_row"] == joined["Fold_person"]
).all():
    raise ValueError("Row/person split assignments disagree.")


# Build fold assignment arrays.

row_to_position = {
    int(r): i
    for i, r in enumerate(aami_df.Row_Index.to_numpy())
}

row_fold["Position"] = (
    row_fold["Row_Index"].map(row_to_position).astype(int)
)

fold_assignment_by_repeat = {}

for repeat in range(1, N_REPEATS + 1):

    current = row_fold[
        row_fold["Repeat"] == repeat
    ]

    assignment = np.full(
        EXPECTED_BEATS,
        -1,
        dtype=np.int8
    )

    assignment[
        current["Position"].to_numpy()
    ] = current["Fold"].to_numpy()

    if not np.isin(assignment, [1, 2, 3, 4, 5]).all():
        raise ValueError(f"Incomplete fold assignment in repeat {repeat}.")

    if current.groupby("Person_ID")["Fold"].nunique().max() != 1:
        raise ValueError(f"Person leakage in repeat {repeat}.")

    record_check = current.merge(
        aami_df[["Row_Index", "Record_ID"]],
        on="Row_Index",
        validate="many_to_one"
    )

    if record_check.groupby("Record_ID")["Fold"].nunique().max() != 1:
        raise ValueError(f"Record leakage in repeat {repeat}.")

    fold_assignment_by_repeat[repeat] = assignment


print("ORIGINAL SPLITS RESTORED — PASS")
print("Split fingerprint:", computed_fp)
print("New splits generated: NO")


# Load Account A's M0 reference OOF file.

reference_oof_path = find_one([
    "m0_corrected_oof_predictions_REFERENCE_MODEL.csv.gz",
    "m0_corrected_oof_predictions_REFERENCE_MODEL.csv"
])

reference_oof = pd.read_csv(
    reference_oof_path,
    compression="infer"
)

print("M0 REFERENCE OOF LOADED")
print("File:", reference_oof_path)
print("Rows:", len(reference_oof))
print("Columns:", reference_oof.columns.tolist())

required_oof = [
    "Repeat", "Fold", "Row_Index",
    "Record_ID", "Person_ID",
    "True_Label", "Predicted_Label"
] + PROB_COLS

missing = [
    c for c in required_oof
    if c not in reference_oof.columns
]

if missing:
    raise ValueError(f"Reference OOF file missing columns: {missing}")


for col in ["Repeat", "Fold", "Row_Index", "Record_ID"]:

    reference_oof[col] = pd.to_numeric(
        reference_oof[col],
        errors="raise"
    ).astype(int)


for col in ["Person_ID", "True_Label", "Predicted_Label"]:

    reference_oof[col] = (
        reference_oof[col].astype(str).str.strip()
    )


if len(reference_oof) != EXPECTED_BEATS * N_REPEATS:
    raise ValueError("Reference OOF row count must be 525,235.")

if reference_oof.duplicated(["Repeat", "Row_Index"]).any():
    raise ValueError("Duplicate reference OOF row/repeat pairs.")


# Check that reference predictions belong to the same beats.

truth = aami_df[
    ["Row_Index", "Record_ID", "Person_ID", "AAMI_Label"]
].rename(columns={"AAMI_Label": "Expected_Label"})

check = reference_oof.merge(
    truth,
    on="Row_Index",
    how="left",
    validate="many_to_one"
)

if check["Expected_Label"].isna().any():
    raise ValueError("Reference OOF has unknown Row_Index values.")

for left, right in [
    ("Record_ID_x", "Record_ID_y"),
    ("Person_ID_x", "Person_ID_y"),
    ("True_Label", "Expected_Label")
]:
    if not (
        check[left].astype(str) == check[right].astype(str)
    ).all():
        raise ValueError(f"Reference OOF mismatch: {left}")

p = reference_oof[PROB_COLS].to_numpy(dtype=float)

if not np.isfinite(p).all():
    raise ValueError("Reference probabilities contain non-finite values.")

if not np.allclose(p.sum(axis=1), 1.0, atol=1e-5):
    raise ValueError("Reference probabilities do not sum to one.")


print("M0 REFERENCE OOF LOADED AND ALIGNED — PASS")
print("Reference predictions:", reference_oof_path)
print("OOF rows:", len(reference_oof))

FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_row_fold_membership.csv
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_person_fold_membership.csv
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_split_fingerprint.txt
ORIGINAL SPLITS RESTORED — PASS
Split fingerprint: 5a07f73452ca78af0b212db5e9f060fc4fcc66f49374c2462249189b847440d9
New splits generated: NO
FOUND: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_corrected_oof_predictions_REFERENCE_MODEL.csv/m0_corrected_oof_predictions_REFERENCE_MODEL.csv
M0 REFERENCE OOF LOADED
File: /kaggle/input/datasets/aakashshah3799/ecg-arrhythmia-project-corrected-v1/m0_corrected_oof_predictions_REFERENCE_MODEL.csv/m0_corrected_oof_predictions_REFERENCE_MODEL.csv
Rows: 525235
Columns: ['Experiment', 'Repeat', 'Fold', 'Row_Index', 'Record_ID', 'Person_ID', 'True_Label', 'Predicted_Label', 'Prob_N', 'Prob_S', 'Prob_

In [4]:
# ---- Load ALL annotation rows (beats + non-beat markers) and verify M1 as a subset ----


def parse_annotation_text(text):
    """Parse rdann-style text: Time, Sample #, Type, Sub, Chan, Num, Aux.

    Lines without an integer Sample # are not event rows. The full annotation
    sequence is retained; no boundary heartbeat annotations are dropped here.
    """
    samples, symbols = [], []
    for line in text.splitlines():
        parts = line.split()
        if len(parts) >= 3 and parts[1].isdigit():
            samples.append(int(parts[1]))
            symbols.append(parts[2])
    return np.asarray(samples, dtype=np.int64), np.asarray(symbols, dtype=object)


def load_annotation_table(record_ids):
    if ANNOTATION_TABLE_PATH is not None:
        t = pd.read_csv(ANNOTATION_TABLE_PATH)
        t = t[["Record_ID", "Sample_Index", "Symbol"]].copy()
        t["Record_ID"] = pd.to_numeric(t["Record_ID"], errors="raise").astype(int)
        t["Sample_Index"] = pd.to_numeric(t["Sample_Index"], errors="raise").astype(np.int64)
        t["Symbol"] = t["Symbol"].astype(str).str.strip()
        return t.sort_values(["Record_ID", "Sample_Index"], kind="stable").reset_index(drop=True)

    frames = []
    for rid in record_ids:
        found = sorted(set(p for p in INPUT_ROOT.rglob(f"{rid}annotations.txt") if p.is_file()))
        if len(found) != 1:
            found = sorted(set(
                p for p in INPUT_ROOT.rglob(f"{rid}*.txt")
                if p.is_file() and "annot" in p.name.lower()
            ))
        if len(found) != 1:
            raise FileNotFoundError(
                f"Record {rid}: expected exactly one annotation text file, found {len(found)}. "
                "Attach the original MIT-BIH annotation files as notebook input, or set "
                "ANNOTATION_TABLE_PATH to a CSV with Record_ID, Sample_Index, Symbol.\n"
                + "\n".join(str(p) for p in found)
            )
        samples, symbols = parse_annotation_text(found[0].read_text(errors="replace"))
        frames.append(pd.DataFrame({
            "Record_ID": int(rid), "Sample_Index": samples, "Symbol": symbols
        }))

    t = pd.concat(frames, ignore_index=True)
    return t.sort_values(["Record_ID", "Sample_Index"], kind="stable").reset_index(drop=True)


ann_table = load_annotation_table(sorted(aami_df["Record_ID"].unique().tolist()))

# 1. Every symbol must be a known beat or a known non-beat marker. No silent guessing.
known = set(BEAT_SYMBOLS) | set(MASK_SYMBOLS) | set(KEEP_SYMBOLS)
unknown = sorted(set(ann_table["Symbol"]) - known)
if unknown:
    raise ValueError(
        f"Annotation symbols not classified as beat or marker: {unknown}. "
        "Decide how to treat them before building RR features."
    )

# Full heartbeat annotation sequence. This intentionally includes annotations
# near recording boundaries, even when their centered 201-sample waveform
# windows cannot be extracted for M1/model use.
ann_beats = (
    ann_table[ann_table["Symbol"].isin(BEAT_SYMBOLS)]
    .sort_values(["Record_ID", "Sample_Index"], kind="stable")
    .reset_index(drop=True)
)

if len(ann_beats) != EXPECTED_FULL_ANNOTATION_BEATS:
    raise ValueError(
        f"Expected {EXPECTED_FULL_ANNOTATION_BEATS} full annotation beats, "
        f"found {len(ann_beats)}. Do not continue until source files are checked."
    )

if ann_beats.duplicated(["Record_ID", "Sample_Index"]).any():
    raise ValueError("Duplicate heartbeat annotation positions found.")

# 2. Verify that every M1/model beat exists exactly once in the full annotation
# sequence with the same original symbol. M1 is expected to be a subset, not an
# equal-sized copy, because 31 boundary annotations have no complete windows.
m1_keys = aami_df[
    ["Record_ID", "Sample_Index", "Original_Symbol"]
].rename(columns={"Original_Symbol": "Symbol"}).copy()

if m1_keys.duplicated(["Record_ID", "Sample_Index"]).any():
    raise ValueError("Duplicate M1 record/sample keys found.")

m1_vs_annotations = m1_keys.merge(
    ann_beats,
    on=["Record_ID", "Sample_Index", "Symbol"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

m1_not_found = m1_vs_annotations[m1_vs_annotations["_merge"] != "both"]
if not m1_not_found.empty:
    raise ValueError(
        "Some M1 beat positions/symbols are not present in the full annotation "
        f"sequence. First rows:\n{m1_not_found.head(10)}"
    )

ann_vs_m1 = ann_beats.merge(
    m1_keys,
    on=["Record_ID", "Sample_Index", "Symbol"],
    how="left",
    indicator=True,
    validate="one_to_one"
)
annotation_only_beats = (
    ann_vs_m1[ann_vs_m1["_merge"] == "left_only"]
    [["Record_ID", "Sample_Index", "Symbol"]]
    .sort_values(["Record_ID", "Sample_Index"], kind="stable")
    .reset_index(drop=True)
)

if len(annotation_only_beats) != EXPECTED_BOUNDARY_EXCLUSIONS:
    raise ValueError(
        f"Expected {EXPECTED_BOUNDARY_EXCLUSIONS} annotation-only boundary "
        f"beats, found {len(annotation_only_beats)}."
    )

# 3. Verify window eligibility using the actual original signal CSVs.
# Complete 201-sample window around sample s requires [s-100, s+100]
# to lie within [0, signal_length-1].

def get_record_signal_length(record_id):
    matches = [
        p for p in INPUT_ROOT.rglob(f"{int(record_id)}.csv")
        if p.is_file() and p.parent.name.lower() == "signals"
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Record {record_id}: expected one original Signals CSV, found {matches}"
        )

    signal_path = matches[0]
    header = pd.read_csv(signal_path, nrows=0)
    clean_to_original = {
        str(c).strip().strip('"').strip("'").strip(): c
        for c in header.columns
    }
    candidates = [
        original for clean, original in clean_to_original.items()
        if clean.lower() in ["sample #", "sample", "sample_index"]
    ]
    if len(candidates) != 1:
        raise ValueError(
            f"Record {record_id}: cannot uniquely identify sample column in {signal_path}"
        )

    sample_col = candidates[0]
    sample_df = pd.read_csv(signal_path, usecols=[sample_col])
    sample_values = pd.to_numeric(sample_df[sample_col], errors="raise").to_numpy(dtype=np.int64)
    if not np.array_equal(sample_values, np.arange(len(sample_values), dtype=np.int64)):
        raise ValueError(f"Record {record_id}: signal sample indices are not 0..N-1.")
    return len(sample_values)


signal_lengths = {
    int(rid): get_record_signal_length(rid)
    for rid in sorted(aami_df["Record_ID"].unique())
}

annotation_only_beats["Signal_Length"] = annotation_only_beats["Record_ID"].map(signal_lengths)
annotation_only_beats["Window_Start"] = annotation_only_beats["Sample_Index"] - 100
annotation_only_beats["Window_End"] = annotation_only_beats["Sample_Index"] + 100
annotation_only_beats["Window_Valid"] = (
    (annotation_only_beats["Window_Start"] >= 0)
    & (annotation_only_beats["Window_End"] < annotation_only_beats["Signal_Length"])
)
annotation_only_beats["Boundary_Reason"] = np.where(
    annotation_only_beats["Window_Start"] < 0,
    "LEFT_RECORDING_BOUNDARY",
    np.where(
        annotation_only_beats["Window_End"] >= annotation_only_beats["Signal_Length"],
        "RIGHT_RECORDING_BOUNDARY",
        "WITHIN_VALID_WINDOW_RANGE"
    )
)

if annotation_only_beats["Window_Valid"].any():
    raise ValueError(
        "At least one annotation-only beat has a complete 201-sample window. "
        "This is not explained by boundary exclusion; inspect these rows:\n"
        + annotation_only_beats[annotation_only_beats["Window_Valid"]].to_string(index=False)
    )

m1_window_valid = (
    (aami_df["Sample_Index"] >= 100)
    & (
        aami_df["Sample_Index"] + 100
        < aami_df["Record_ID"].map(signal_lengths).to_numpy()
    )
)
if not bool(m1_window_valid.all()):
    bad = aami_df.loc[~m1_window_valid, ["Record_ID", "Sample_Index", "Original_Symbol"]]
    raise ValueError(
        "At least one M1/model beat has an incomplete 201-sample window:\n"
        + bad.head(20).to_string(index=False)
    )

# The full sequence is now verified as the RR source. The 31 boundary rows
# remain in ann_beats and will contribute to neighbouring interval calculations.
ANNOTATION_BEATS_MATCH_M1 = True

# 4. Non-beat markers are retained separately for interval QC.
markers = ann_table[~ann_table["Symbol"].isin(BEAT_SYMBOLS)].copy()
markers_by_record = {int(r): g for r, g in markers.groupby("Record_ID")}

beat_keys = set(zip(ann_beats["Record_ID"].to_numpy(), ann_beats["Sample_Index"].to_numpy()))
coincide = sum(
    (r, s) in beat_keys
    for r, s in zip(markers["Record_ID"], markers["Sample_Index"])
)

print("FULL ANNOTATION SEQUENCE VERIFIED — PASS")
print("All annotation rows:", len(ann_table))
print("Full heartbeat annotations for RR calculations:", len(ann_beats))
print("M1/model beats matched exactly:", len(m1_keys))
print("Annotation-only boundary annotations:", len(annotation_only_beats))
print("Boundary exclusion reasons:", annotation_only_beats["Boundary_Reason"].value_counts().to_dict())
print("All 31 annotation-only beats lack complete windows: PASS")
print("All 105,047 M1/model windows are complete: PASS")
print("Non-beat marker rows:", len(markers))
print("Marker counts by symbol:", markers["Symbol"].value_counts().to_dict())
print("Markers sharing a sample index with a beat (not treated as inside):", coincide)

print("\nANNOTATION-ONLY BOUNDARY BEATS (excluded from model rows, retained for RR timing)")
display(annotation_only_beats)
annotation_only_beats.to_csv(
    OUTPUT_DIR / "m2_annotation_boundary_exclusions.csv", index=False
)
print("Boundary-exclusion audit saved:", OUTPUT_DIR / "m2_annotation_boundary_exclusions.csv")


FULL ANNOTATION SEQUENCE VERIFIED — PASS
All annotation rows: 108144
Full heartbeat annotations for RR calculations: 105078
M1/model beats matched exactly: 105047
Annotation-only boundary annotations: 31
Boundary exclusion reasons: {'LEFT_RECORDING_BOUNDARY': 17, 'RIGHT_RECORDING_BOUNDARY': 14}
All 31 annotation-only beats lack complete windows: PASS
All 105,047 M1/model windows are complete: PASS
Non-beat marker rows: 3066
Marker counts by symbol: {'+': 1241, '~': 579, '!': 472, '"': 437, 'x': 193, '|': 132, '[': 6, ']': 6}
Markers sharing a sample index with a beat (not treated as inside): 0

ANNOTATION-ONLY BOUNDARY BEATS (excluded from model rows, retained for RR timing)


,Record_ID,Sample_Index,Symbol,Signal_Length,Window_Start,Window_End,Window_Valid,Boundary_Reason
0,100,77,N,650000,-23,177,False,LEFT_RECORDING_BOUNDARY
1,100,649991,N,650000,649891,650091,False,RIGHT_RECORDING_BOUNDARY
2,101,83,N,650000,-17,183,False,LEFT_RECORDING_BOUNDARY
3,107,95,/,650000,-5,195,False,LEFT_RECORDING_BOUNDARY
4,108,88,N,650000,-12,188,False,LEFT_RECORDING_BOUNDARY
5,109,649931,L,650000,649831,650031,False,RIGHT_RECORDING_BOUNDARY
6,113,649994,N,650000,649894,650094,False,RIGHT_RECORDING_BOUNDARY
7,115,649955,N,650000,649855,650055,False,RIGHT_RECORDING_BOUNDARY
8,116,649957,N,650000,649857,650057,False,RIGHT_RECORDING_BOUNDARY
9,117,649926,N,650000,649826,650026,False,RIGHT_RECORDING_BOUNDARY


Boundary-exclusion audit saved: /kaggle/working/ECG_Project/05_M2_RR_FEATURES_VERSION_B/m2_annotation_boundary_exclusions.csv


In [5]:
# ---- Pure RR functions (Version B) ----


def interval_flags(beat_samples, marker_samples, marker_symbols, mask_symbols=MASK_SYMBOLS):
    """Return flags indexed by beat position: flag[j] masks beat j-1 -> beat j.

    Only non-beat marker samples strictly inside an interval are considered.
    A marker exactly on an endpoint is not inside that interval.
    """
    s = np.asarray(beat_samples, dtype=np.int64)
    n = len(s)
    flag = np.zeros(n, dtype=bool)
    if n < 2 or len(marker_samples) == 0:
        return flag

    m_pos = np.asarray(marker_samples, dtype=np.int64)
    m_sym = np.asarray(marker_symbols, dtype=object)
    order = np.argsort(m_pos, kind="stable")
    m_pos, m_sym = m_pos[order], m_sym[order]

    is_mask = np.isin(m_sym, list(mask_symbols)).astype(np.int64)
    cumulative = np.concatenate([[0], np.cumsum(is_mask)])
    left = np.searchsorted(m_pos, s[:-1], side="right")
    right = np.searchsorted(m_pos, s[1:], side="left")
    flag[1:] = (cumulative[right] - cumulative[left]) > 0
    return flag


def compute_rr_features(beat_samples, flagged=None, fs=SAMPLE_RATE_HZ,
                        window=RR_WINDOW, min_valid=RR_MIN_VALID):
    """Return [pre_RR, post_RR, strict previous-window mean, pre/mean, post/pre].

    For Version B, min_valid == window == 10: every one of the ten immediately
    preceding interval positions must be valid. The current pre_RR is excluded
    from its own historical reference. Masked intervals propagate as NaN.
    """
    s = np.asarray(beat_samples, dtype=np.int64)
    n = len(s)
    if n < 2:
        raise ValueError("Need at least two beats.")
    if window != 10 or min_valid != 10:
        raise ValueError("Version B requires a strict 10-of-10 historical window.")

    gaps = np.diff(s)
    if np.any(gaps <= 0):
        raise ValueError("Beat positions must be strictly increasing.")

    val = np.full(n, np.nan, dtype=np.float64)  # val[j]: interval ending at beat j
    val[1:] = gaps / fs
    if flagged is not None:
        flagged = np.asarray(flagged, dtype=bool)
        if flagged.shape != (n,):
            raise ValueError("flagged must have one entry per beat position.")
        val[flagged] = np.nan

    pre = val.copy()
    post = np.full(n, np.nan, dtype=np.float64)
    post[:-1] = val[1:]

    # Shift by one so the current interval is never included in its own history.
    mean_prev = (
        pd.Series(pre)
        .shift(1)
        .rolling(window=window, min_periods=window)
        .mean()
        .to_numpy(dtype=np.float64)
    )

    with np.errstate(divide="ignore", invalid="ignore"):
        ratio_pre = np.where(
            np.isfinite(pre) & np.isfinite(mean_prev) & (mean_prev > 0),
            pre / mean_prev,
            np.nan
        )
        ratio_post = np.where(
            np.isfinite(post) & np.isfinite(pre) & (pre > 0),
            post / pre,
            np.nan
        )

    return np.column_stack([pre, post, mean_prev, ratio_pre, ratio_post])


# ---- Synthetic self-tests: strict 10-of-10 policy and mask propagation ----

_s = np.arange(0, 360 * 50, 360)
_n = len(_s)

_X = compute_rr_features(_s)
assert np.isnan(_X[0, 0]) and np.isnan(_X[-1, 1])
# pre_RR[0] is the recording boundary; first 10 historical windows are not full.
assert np.isnan(_X[:11, 2]).all()
assert np.isfinite(_X[11:, 2]).all()
assert np.allclose(_X[1:, 0], 1.0)
assert np.allclose(_X[11:, 2], 1.0)

_fl = np.zeros(_n, dtype=bool)
_fl[20] = True  # mask interval beat 19 -> beat 20
_Y = compute_rr_features(_s, _fl)
_nan = lambda col: set(np.flatnonzero(np.isnan(_Y[:, col])).tolist())
assert _nan(0) == {0, 20}
assert _nan(1) == {19, _n - 1}
# Historical mean is missing until 10 complete intervals exist, then again
# for rows whose preceding 10-interval window includes masked interval 20.
assert _nan(2) == set(range(0, 11)) | set(range(21, 31))
assert _nan(3) == set(range(0, 11)) | set(range(20, 31))
assert _nan(4) == {0, 19, 20, _n - 1}

_beats = np.array([100, 460, 820, 1180, 1540])
_flags = interval_flags(
    _beats,
    [700, 1000, 1300, 1180],
    ["x", "~", "[", "~"]
)
assert _flags.tolist() == [False, False, False, True, True]
assert not interval_flags(_beats, [], []).any()

print("VERSION B RR FUNCTION SELF-TESTS — PASS")
print("Strict previous-10 requirement: 10 valid out of 10")
print("Current interval excluded from its own historical mean: PASS")
print("Masked interval propagation: PASS")


VERSION B RR FUNCTION SELF-TESTS — PASS
Strict previous-10 requirement: 10 valid out of 10
Current interval excluded from its own historical mean: PASS
Masked interval propagation: PASS


In [6]:
# ---- Build Version B RR_FEATURES from all original annotations ----
# RR timing uses all 105,078 annotated heartbeat positions.
# Only the 105,047 M1/model rows receive model features.

rr_matrix = np.full((EXPECTED_BEATS, len(RR_FEATURES)), np.nan, dtype=np.float64)
assigned = np.zeros(EXPECTED_BEATS, dtype=bool)

audit_rows = []
interval_qc_rows = []
candidate4_hits = []
post_ivr_hits = []
long_total = 0
long_masked = 0
max_unmasked_interval = 0.0
raw_pipe_marker_count = int((ann_table["Symbol"] == "|").sum())
pipe_markers_inside_count = 0

MASK_REASON_LABELS = {
    "~": "SIGNAL_QUALITY_MARKER",
    "|": "ISOLATED_QRS_ARTIFACT_MARKER",
    "!": "VENTRICULAR_FLUTTER_WAVE_MARKER",
    "[": "VFL_EPISODE_START_MARKER",
    "]": "VFL_EPISODE_END_MARKER"
}

for record_id, full_group in ann_beats.groupby("Record_ID", sort=True):

    full_group = full_group.sort_values("Sample_Index", kind="stable")
    samples = full_group["Sample_Index"].to_numpy(dtype=np.int64)
    full_symbols = full_group["Symbol"].to_numpy(dtype=object)

    if len(samples) < 2 or np.any(np.diff(samples) <= 0):
        raise ValueError(
            f"Record {record_id}: full annotation beat positions are not strictly increasing."
        )

    mk = markers_by_record.get(int(record_id))
    if mk is None:
        mk_pos = np.array([], dtype=np.int64)
        mk_sym = np.array([], dtype=object)
    else:
        mk = mk.sort_values("Sample_Index", kind="stable")
        mk_pos = mk["Sample_Index"].to_numpy(dtype=np.int64)
        mk_sym = mk["Symbol"].to_numpy(dtype=object)

    # Marker-driven mask, based only on markers strictly inside each interval.
    marker_flagged = interval_flags(samples, mk_pos, mk_sym)

    # Candidate 4: exactly one documented interval-level QC exception.
    candidate4_flagged = np.zeros(len(samples), dtype=bool)
    if int(record_id) == CANDIDATE4_INTERVAL[0]:
        candidate_matches = np.flatnonzero(
            (samples[:-1] == CANDIDATE4_INTERVAL[1])
            & (samples[1:] == CANDIDATE4_INTERVAL[2])
        )
        if len(candidate_matches) != 1:
            raise ValueError(
                "Candidate 4 interval must occur exactly once in Record 232; "
                f"found {len(candidate_matches)}."
            )
        candidate_index = int(candidate_matches[0] + 1)
        if marker_flagged[candidate_index]:
            raise ValueError(
                "Candidate 4 unexpectedly also contains a marker-mask trigger. "
                "Reconcile the raw annotations before continuing."
            )
        candidate4_flagged[candidate_index] = True
        candidate4_hits.append((int(record_id), int(samples[candidate_index - 1]), int(samples[candidate_index])))

    # General marker rules take precedence; Candidate 4 is an additional flag.
    flagged = marker_flagged | candidate4_flagged
    full_block = compute_rr_features(samples, flagged)

    # Create a complete interval-level QC sidecar (one row for every adjacent beat pair).
    seconds = np.diff(samples) / SAMPLE_RATE_HZ
    for j in range(len(samples) - 1):
        start_sample = int(samples[j])
        end_sample = int(samples[j + 1])
        interval_id = f"{int(record_id)}_{start_sample}_{end_sample}"

        inside_mask = (mk_pos > start_sample) & (mk_pos < end_sample)
        inside_symbols = mk_sym[inside_mask]
        inside_mask_symbols = [str(x) for x in inside_symbols if str(x) in MASK_SYMBOLS]
        inside_context_symbols = [str(x) for x in inside_symbols if str(x) in KEEP_SYMBOLS]

        reasons = []
        for symbol in MASK_SYMBOLS:
            if symbol in inside_mask_symbols:
                reasons.append(MASK_REASON_LABELS[symbol])

        candidate4_here = bool(candidate4_flagged[j + 1])
        if candidate4_here:
            reasons.append("UNRESOLVED_TWO_LEAD_WAVEFORM_CANDIDATE_4")

        post_ivr_here = (
            int(record_id), start_sample, end_sample
        ) in KNOWN_POST_IVR_INTERVALS

        if post_ivr_here:
            if str(full_symbols[j]) != "E" or str(full_symbols[j + 1]) != "E":
                raise ValueError(
                    f"Known post-IVR interval {interval_id} does not have E-to-E heartbeat endpoints."
                )
            post_ivr_hits.append((int(record_id), start_sample, end_sample))

        is_masked = bool(flagged[j + 1])
        if post_ivr_here and is_masked:
            # This is allowed only if a real internal mask trigger exists.
            if not marker_flagged[j + 1]:
                raise ValueError("Unexpected post-IVR mask without an internal marker.")

        pipe_inside = int(np.sum(inside_symbols == "|"))
        pipe_markers_inside_count += pipe_inside

        interval_qc_rows.append({
            "Interval_ID": interval_id,
            "Record_ID": int(record_id),
            "Start_Sample": start_sample,
            "End_Sample": end_sample,
            "Duration_sec": float(seconds[j]),
            "Previous_Beat_Symbol": str(full_symbols[j]),
            "Next_Beat_Symbol": str(full_symbols[j + 1]),
            "Symbols_Inside": ";".join(str(x) for x in inside_symbols),
            "Mask_Markers_Inside": ";".join(inside_mask_symbols),
            "Context_Markers_Inside": ";".join(inside_context_symbols),
            "Mask_Reason": ";".join(reasons) if reasons else "NONE",
            "Marker_Masked": bool(marker_flagged[j + 1]),
            "Candidate4_Explicit_QC_Mask": candidate4_here,
            "Post_IVR_E_TO_E": bool(post_ivr_here),
            "Pipe_Markers_Inside_Count": pipe_inside,
            "Is_Masked": is_masked
        })

    # Select only M1/model beats using composite sample-position alignment.
    target_group = aami_df[aami_df["Record_ID"] == int(record_id)].copy()
    target_group = target_group.sort_values("Sample_Index", kind="stable")
    target_samples = target_group["Sample_Index"].to_numpy(dtype=np.int64)
    target_rows = target_group["Row_Index"].to_numpy(dtype=np.int64)
    target_symbols = target_group["Original_Symbol"].to_numpy(dtype=object)

    full_indices = np.searchsorted(samples, target_samples)
    if np.any(full_indices >= len(samples)):
        raise ValueError(f"Record {record_id}: an M1 beat is absent from full annotations.")
    if not np.array_equal(samples[full_indices], target_samples):
        raise ValueError(f"Record {record_id}: sample-key alignment failed.")
    if not np.array_equal(full_symbols[full_indices], target_symbols):
        raise ValueError(f"Record {record_id}: original-symbol alignment failed.")
    if assigned[target_rows].any():
        raise ValueError(f"Record {record_id}: model Row_Index assigned more than once.")

    target_block = full_block[full_indices]

    # Independent expected missingness audit for the entire original beat sequence.
    expected_pre_missing = np.zeros(len(samples), dtype=bool)
    expected_pre_missing[0] = True
    expected_pre_missing[1:] = flagged[1:]

    expected_post_missing = np.zeros(len(samples), dtype=bool)
    expected_post_missing[-1] = True
    expected_post_missing[:-1] = flagged[1:]

    pre_full = full_block[:, 0]
    expected_mean_missing = np.ones(len(samples), dtype=bool)
    for i in range(len(samples)):
        history_start = max(0, i - RR_WINDOW)
        history = pre_full[history_start:i]
        expected_mean_missing[i] = not (
            len(history) == RR_WINDOW and np.isfinite(history).all()
        )

    expected_ratio_pre_missing = expected_pre_missing | expected_mean_missing
    expected_ratio_post_missing = expected_post_missing | expected_pre_missing

    expected_nan_masks = np.column_stack([
        expected_pre_missing,
        expected_post_missing,
        expected_mean_missing,
        expected_ratio_pre_missing,
        expected_ratio_post_missing
    ])[full_indices]

    if not np.array_equal(np.isnan(target_block), expected_nan_masks):
        raise ValueError(
            f"Record {record_id}: Version B missing-value propagation failed."
        )

    rr_matrix[target_rows, :] = target_block
    assigned[target_rows] = True

    flag_intervals = flagged[1:]
    marker_flag_intervals = marker_flagged[1:]
    candidate4_intervals = candidate4_flagged[1:]
    is_long = seconds > 2.0
    long_total += int(is_long.sum())
    long_masked += int((is_long & flag_intervals).sum())

    unmasked = seconds[~flag_intervals]
    if len(unmasked):
        max_unmasked_interval = max(max_unmasked_interval, float(unmasked.max()))

    audit_rows.append({
        "Record_ID": int(record_id),
        "Full_Annotation_Beats": int(len(samples)),
        "M1_Model_Beats": int(len(target_samples)),
        "Marker_Masked_Intervals": int(marker_flag_intervals.sum()),
        "Candidate4_Explicitly_Masked_Intervals": int(candidate4_intervals.sum()),
        "Total_Masked_Intervals": int(flag_intervals.sum()),
        "Post_IVR_E_TO_E_Intervals": int(sum(
            (int(record_id), int(samples[k]), int(samples[k + 1])) in KNOWN_POST_IVR_INTERVALS
            for k in range(len(samples) - 1)
        )),
        "Pipe_Markers_In_Intervals": int(sum(
            row["Pipe_Markers_Inside_Count"]
            for row in interval_qc_rows
            if row["Record_ID"] == int(record_id)
        )),
        "Median_Interval_sec": float(np.median(seconds)),
        "Max_Interval_sec": float(seconds.max()),
        "Max_Unmasked_Interval_sec": float(unmasked.max()) if len(unmasked) else np.nan,
        "Model_Rows_With_Any_Missing_RR": int(np.isnan(target_block).any(axis=1).sum())
    })

if not assigned.all():
    missing_rows = np.flatnonzero(~assigned)
    raise ValueError(
        f"RR features were not assigned to {len(missing_rows)} model rows. "
        f"First Row_Index values: {missing_rows[:20].tolist()}"
    )

if len(candidate4_hits) != 1 or candidate4_hits[0] != CANDIDATE4_INTERVAL:
    raise ValueError(f"Candidate 4 QC decision was not applied exactly once: {candidate4_hits}")

if set(post_ivr_hits) != KNOWN_POST_IVR_INTERVALS or len(post_ivr_hits) != 2:
    raise ValueError(f"Post-IVR E-to-E context intervals were not identified exactly: {post_ivr_hits}")

interval_qc = pd.DataFrame(interval_qc_rows)
if len(interval_qc) != len(ann_beats) - EXPECTED_RECORDS:
    raise ValueError(
        f"Unexpected interval sidecar row count: {len(interval_qc)}; "
        f"expected {len(ann_beats) - EXPECTED_RECORDS}."
    )
if interval_qc["Interval_ID"].duplicated().any():
    raise ValueError("Duplicate interval IDs in Version B interval QC sidecar.")

marker_masked_intervals = interval_qc[interval_qc["Marker_Masked"]].copy()
masked_intervals = interval_qc[interval_qc["Is_Masked"]].copy()

n_marker_masked = len(marker_masked_intervals)
n_masked = len(masked_intervals)
pipe_intervals = interval_qc[interval_qc["Pipe_Markers_Inside_Count"] > 0]
pipe_unmasked_intervals = pipe_intervals[~pipe_intervals["Is_Masked"]]

# Reconcile the raw '|' annotations independently rather than relying on the
# older audit's precomputed Has_Artifact_Marker field.
if len(pipe_unmasked_intervals) != 0:
    raise ValueError(
        "At least one interval containing an internal '|' marker was not masked."
    )
if n_marker_masked != EXPECTED_VERSION_A_MARKER_MASKED_INTERVALS:
    raise ValueError(
        f"Marker-based mask count changed: expected {EXPECTED_VERSION_A_MARKER_MASKED_INTERVALS}, "
        f"found {n_marker_masked}. Reconcile the raw annotation symbols (especially '|') "
        "before training Version B."
    )
if n_masked != EXPECTED_VERSION_B_MASKED_INTERVALS:
    raise ValueError(
        f"Expected {EXPECTED_VERSION_B_MASKED_INTERVALS} total Version B masked intervals, "
        f"found {n_masked}. Do not train until reconciled."
    )

# Candidate 4 must be the only additional mask relative to the marker rule.
if int(interval_qc["Candidate4_Explicit_QC_Mask"].sum()) != 1:
    raise ValueError("Candidate 4 mask count must equal exactly one.")
if int((interval_qc["Is_Masked"] & ~interval_qc["Marker_Masked"]).sum()) != 1:
    raise ValueError("Version B must have exactly one non-marker mask: Candidate 4.")

# The two post-IVR intervals should remain visible and numeric unless marker-masked.
post_ivr_rows = interval_qc[interval_qc["Post_IVR_E_TO_E"]]
if len(post_ivr_rows) != 2 or post_ivr_rows["Is_Masked"].any():
    raise ValueError("The two known post-IVR E-to-E intervals were not retained as specified.")

RR_NAN_COUNTS_MATCH_POLICY = True
masked_intervals = masked_intervals.reset_index(drop=True)
rr_audit = pd.DataFrame(audit_rows).sort_values("Record_ID").reset_index(drop=True)

# ---- Validation on the 105,047 M1/model rows ----
assert rr_matrix.shape == (EXPECTED_BEATS, len(RR_FEATURES))
assert not np.isinf(rr_matrix).any(), "Infinite RR value."
finite_values = rr_matrix[np.isfinite(rr_matrix)]
assert (finite_values > 0).all(), "Non-positive RR value."

pre_nan = int(np.isnan(rr_matrix[:, 0]).sum())
post_nan = int(np.isnan(rr_matrix[:, 1]).sum())
mean_nan = int(np.isnan(rr_matrix[:, 2]).sum())
ratio_pre_nan = int(np.isnan(rr_matrix[:, 3]).sum())
ratio_post_nan = int(np.isnan(rr_matrix[:, 4]).sum())

NO_UNMASKED_INTERVAL_OVER_60S = max_unmasked_interval <= 60.0
if not NO_UNMASKED_INTERVAL_OVER_60S:
    raise ValueError(
        f"An unmasked interval of {max_unmasked_interval:.1f} s exists. "
        "Inspect event markers before continuing."
    )

any_missing = np.isnan(rr_matrix).any(axis=1)
rr_feature_df = pd.DataFrame(rr_matrix, columns=RR_FEATURES)
rr_feature_df.insert(0, "Row_Index", aami_df["Row_Index"].to_numpy())
rr_feature_df.insert(1, "Record_ID", aami_df["Record_ID"].to_numpy())
rr_feature_df.insert(2, "Person_ID", aami_df["Person_ID"].to_numpy())
rr_feature_df.insert(3, "Sample_Index", aami_df["Sample_Index"].to_numpy())
rr_feature_df.insert(4, "AAMI_Label", aami_df["AAMI_Label"].to_numpy())

# Exact one-to-one composite-key validation for the saved Version B feature table.
expected_keys = aami_df[["Record_ID", "Sample_Index", "Original_Symbol", "Row_Index"]].copy()
feature_keys = rr_feature_df[["Record_ID", "Sample_Index", "Row_Index"]].copy()
key_check = expected_keys.merge(
    feature_keys,
    on=["Record_ID", "Sample_Index", "Row_Index"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)
if not (key_check["_merge"] == "both").all():
    raise ValueError("Version B RR table does not match every M1 composite key exactly once.")

if len(rr_feature_df) != EXPECTED_BEATS:
    raise ValueError("Version B changed the number of M1/model rows.")
if rr_feature_df[["Record_ID", "Sample_Index"]].duplicated().any():
    raise ValueError("Duplicate Version B composite keys found.")
if rr_feature_df[["Record_ID", "Sample_Index"]].isna().any().any():
    raise ValueError("Missing Version B composite key values found.")

sample_order_check = (
    rr_feature_df.groupby("Record_ID", sort=False)["Sample_Index"]
    .apply(lambda values: values.is_monotonic_increasing)
)
if len(sample_order_check) != EXPECTED_RECORDS or not sample_order_check.all():
    raise ValueError(
        "Sample positions are not strictly increasing within every recording: "
        f"{sample_order_check[~sample_order_check].index.tolist()}"
    )

RR_FEATURE_ROWS_PRESERVED = len(rr_feature_df) == EXPECTED_BEATS
RR_FEATURE_KEYS_UNIQUE = not rr_feature_df[["Record_ID", "Sample_Index"]].duplicated().any()
RR_SAMPLE_ORDER_VALID = bool(sample_order_check.all())

print("VERSION B RR_FEATURES CREATED — PASS")
print("Feature rows preserved:", len(rr_feature_df), "/", EXPECTED_BEATS)
print("Duplicate composite keys:", int(rr_feature_df[["Record_ID", "Sample_Index"]].duplicated().sum()))
print("Missing composite key values:", int(rr_feature_df[["Record_ID", "Sample_Index"]].isna().sum().sum()))
print("Recordings with increasing sample positions:", int(sample_order_check.sum()), "/", EXPECTED_RECORDS)
print("Full annotated beats used for timing:", len(ann_beats))
print("Model rows retained:", len(aami_df))
print("RR feature matrix shape:", rr_matrix.shape)
print("Boundary annotations included in timing only:", len(annotation_only_beats))
print("Raw '|' annotation rows:", raw_pipe_marker_count)
print("'|' markers strictly inside RR intervals:", pipe_markers_inside_count)
print("Distinct intervals containing internal '|':", len(pipe_intervals))
print("Unmasked intervals containing internal '|':", len(pipe_unmasked_intervals))
print("Marker-masked intervals:", n_marker_masked)
print("Candidate 4 explicitly masked:", candidate4_hits)
print("Total Version B masked intervals:", n_masked)
print("Post-IVR E-to-E intervals retained:", len(post_ivr_rows))
print("Intervals longer than 2 s:", long_total, "| of which masked:", long_masked)
print("Longest UNMASKED interval (s):", round(max_unmasked_interval, 3))
print("Missing counts per feature:", dict(zip(RR_FEATURES, np.isnan(rr_matrix).sum(axis=0).tolist())))
print("Rows with any missing RR value:", int(any_missing.sum()),
      f"({100 * any_missing.mean():.2f}%)")
print("Missing RR rows by AAMI class:",
      pd.Series(aami_df["AAMI_Label"].to_numpy()[any_missing]).value_counts()
      .reindex(CLASS_ORDER, fill_value=0).to_dict())
print("pre_RR NaN:", pre_nan, "| post_RR NaN:", post_nan)
print("mean_previous_10_RR NaN:", mean_nan)
print("pre-RR ratio NaN:", ratio_pre_nan, "| post/pre ratio NaN:", ratio_post_nan)
print("Composite key alignment: one-to-one PASS")

print("\nVERSION B RR FEATURE SUMMARY (finite values only)")
display(rr_feature_df[RR_FEATURES].describe().T)

print("\nVERSION B MASKED INTERVALS")
display(masked_intervals)

print("\nVERSION B PER-RECORD AUDIT")
display(rr_audit)

print("\nPOST-IVR CONTEXT CHECK")
display(post_ivr_rows[["Interval_ID", "Duration_sec", "Previous_Beat_Symbol", "Next_Beat_Symbol", "Is_Masked", "Post_IVR_E_TO_E"]])

rr_feature_df.to_csv(OUTPUT_DIR / "m2_rr_features_version_b.csv.gz", index=False, compression="gzip")
rr_audit.to_csv(OUTPUT_DIR / "m2_rr_per_record_audit_version_b.csv", index=False)
interval_qc.to_csv(OUTPUT_DIR / "m2_rr_interval_qc_flags_version_b.csv.gz", index=False, compression="gzip")
masked_intervals.to_csv(OUTPUT_DIR / "m2_rr_masked_intervals_version_b.csv", index=False)
rr_feature_df[RR_FEATURES].describe().T.to_csv(OUTPUT_DIR / "m2_rr_feature_summary_version_b.csv")

print("\nVersion B RR feature table, interval QC sidecar, mask list, summary and per-record audit saved.")


VERSION B RR_FEATURES CREATED — PASS
Feature rows preserved: 105047 / 105047
Duplicate composite keys: 0
Missing composite key values: 0
Recordings with increasing sample positions: 46 / 46
Full annotated beats used for timing: 105078
Model rows retained: 105047
RR feature matrix shape: (105047, 5)
Boundary annotations included in timing only: 31
Raw '|' annotation rows: 132
'|' markers strictly inside RR intervals: 132
Distinct intervals containing internal '|': 120
Unmasked intervals containing internal '|': 0
Marker-masked intervals: 687
Candidate 4 explicitly masked: [(232, 629048, 629972)]
Total Version B masked intervals: 688
Post-IVR E-to-E intervals retained: 2
Intervals longer than 2 s: 141 | of which masked: 10
Longest UNMASKED interval (s): 5.872
Missing counts per feature: {'pre_RR': 717, 'post_RR': 720, 'mean_previous_10_RR': 5741, 'pre_RR_div_mean_previous_10_RR': 6145, 'post_RR_div_pre_RR': 1387}
Rows with any missing RR value: 6554 (6.24%)
Missing RR rows by AAMI class:

,count,mean,std,min,25%,50%,75%,max
pre_RR,104330.0,0.788886,0.225815,0.250000,0.650000,0.752778,0.897222,5.872222
post_RR,104327.0,0.788885,0.225809,0.250000,0.650000,0.752778,0.897222,5.872222
mean_previous_10_RR,99306.0,0.790830,0.180481,0.316667,0.663056,0.756389,0.889167,1.958611
pre_RR_div_mean_previous_10_RR,98902.0,1.002760,0.185755,0.281149,0.960878,0.997279,1.032922,7.788754
post_RR_div_pre_RR,103660.0,1.034364,0.311989,0.202933,0.959514,0.996241,1.032544,7.687273



VERSION B MASKED INTERVALS


,Interval_ID,Record_ID,Start_Sample,End_Sample,Duration_sec,Previous_Beat_Symbol,Next_Beat_Symbol,Symbols_Inside,Mask_Markers_Inside,Context_Markers_Inside,Mask_Reason,Marker_Masked,Candidate4_Explicit_QC_Mask,Post_IVR_E_TO_E,Pipe_Markers_Inside_Count,Is_Masked
0,101_40340_40668,101,40340,40668,0.911111,N,N,~,~,,SIGNAL_QUALITY_MARKER,True,False,False,0,True
1,101_40668_40976,101,40668,40976,0.855556,N,N,|,|,,ISOLATED_QRS_ARTIFACT_MARKER,True,False,False,1,True
2,101_41829_42126,101,41829,42126,0.825000,N,N,~,~,,SIGNAL_QUALITY_MARKER,True,False,False,0,True
3,101_62994_63315,101,62994,63315,0.891667,N,N,|,|,,ISOLATED_QRS_ARTIFACT_MARKER,True,False,False,1,True
4,101_63315_63631,101,63315,63631,0.877778,N,N,|,|,,ISOLATED_QRS_ARTIFACT_MARKER,True,False,False,1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
683,234_147302_147541,234,147302,147541,0.663889,N,N,~,~,,SIGNAL_QUALITY_MARKER,True,False,False,0,True
684,234_486337_486572,234,486337,486572,0.652778,N,N,~,~,,SIGNAL_QUALITY_MARKER,True,False,False,0,True
685,234_487295_487534,234,487295,487534,0.663889,N,N,~,~,,SIGNAL_QUALITY_MARKER,True,False,False,0,True
686,234_493581_493822,234,493581,493822,0.669444,N,N,~,~,,SIGNAL_QUALITY_MARKER,True,False,False,0,True



VERSION B PER-RECORD AUDIT


,Record_ID,Full_Annotation_Beats,M1_Model_Beats,Marker_Masked_Intervals,Candidate4_Explicitly_Masked_Intervals,Total_Masked_Intervals,Post_IVR_E_TO_E_Intervals,Pipe_Markers_In_Intervals,Median_Interval_sec,Max_Interval_sec,Max_Unmasked_Interval_sec,Model_Rows_With_Any_Missing_RR
0,100,2273,2271,0,0,0,0,0,0.797222,1.130556,1.130556,10
1,101,1865,1864,7,0,7,0,4,0.977778,1.888889,1.275000,55
2,103,2084,2084,6,0,6,0,0,0.869444,1.116667,1.116667,84
3,105,2572,2572,114,0,114,0,30,0.702778,1.430556,1.063889,902
4,106,2027,2027,29,0,29,0,0,0.936111,1.502778,1.502778,340
5,107,2137,2136,2,0,2,0,0,0.847222,0.944444,0.944444,26
6,108,1763,1762,48,0,48,0,8,1.027778,1.711111,1.711111,449
7,109,2532,2531,2,0,2,0,0,0.713889,0.919444,0.919444,35
8,111,2124,2124,8,0,8,0,0,0.852778,1.025000,1.025000,81
9,112,2539,2539,10,0,10,0,0,0.711111,0.963889,0.963889,109



POST-IVR CONTEXT CHECK


,Interval_ID,Duration_sec,Previous_Beat_Symbol,Next_Beat_Symbol,Is_Masked,Post_IVR_E_TO_E
57185,207_590585_591405,2.277778,E,E,False,True
57186,207_591405_592308,2.508333,E,E,False,True



Version B RR feature table, interval QC sidecar, mask list, summary and per-record audit saved.


In [7]:
# Recreate REFERENCE_MODEL features exactly as in M0.
wave = aami_df[
    [f"MLII_{i}" for i in range(201)]
].to_numpy(dtype=np.float32)

means = wave.mean(axis=1, keepdims=True)

stds = wave.std(axis=1, keepdims=True)

stds[stds == 0] = 1.0

normalized = (wave - means) / stds

d1 = np.diff(normalized, axis=1)

d2 = np.diff(normalized, n=2, axis=1)

q25, q50, q75 = np.percentile(
    normalized,
    [25, 50, 75],
    axis=1
)


REFERENCE_FEATURE_NAMES = [
    "d1_mean",
    "d1_std",
    "d1_max",
    "d1_min",
    "d1_abs_mean",
    "d1_energy",
    "d2_std",
    "d2_abs_mean",
    "normalized_median",
    "normalized_IQR",
    "normalized_q25",
    "normalized_q75"
]


X_REFERENCE = np.column_stack([
    d1.mean(axis=1),
    d1.std(axis=1),
    d1.max(axis=1),
    d1.min(axis=1),
    np.abs(d1).mean(axis=1),
    np.square(d1).mean(axis=1),
    d2.std(axis=1),
    np.abs(d2).mean(axis=1),
    q50,
    q75 - q25,
    q25,
    q75
]).astype(np.float32)


# RR features keep their missing values (NaN). Random Forest handles NaN natively.
X_RR = rr_matrix.astype(np.float32)

X_REFERENCE_PLUS_RR = np.column_stack([
    X_REFERENCE,
    X_RR
]).astype(np.float32)


y = aami_df["AAMI_Label"].to_numpy(dtype=str)

groups = aami_df["Person_ID"].to_numpy(dtype=str)

record_ids = aami_df["Record_ID"].to_numpy(dtype=np.int64)

row_ids = aami_df["Row_Index"].to_numpy(dtype=np.int64)


# Matrix shape checks.

assert X_REFERENCE.shape == (EXPECTED_BEATS, 12)

assert X_RR.shape == (EXPECTED_BEATS, 5)

assert X_REFERENCE_PLUS_RR.shape == (EXPECTED_BEATS, 17)

assert np.isfinite(X_REFERENCE).all()

assert not np.isinf(X_RR).any()

assert np.isfinite(X_REFERENCE_PLUS_RR[:, :12]).all()

assert np.array_equal(
    np.isnan(X_REFERENCE_PLUS_RR[:, 12:]),
    np.isnan(X_RR)
)

# Native NaN support in Random Forest needs scikit-learn >= 1.4.
_sk = tuple(int(v) for v in sklearn.__version__.split(".")[:2])
assert _sk >= (1, 4), f"scikit-learn {sklearn.__version__} cannot fit Random Forest with NaN."


feature_checks = pd.DataFrame([
    {
        "Matrix": "REFERENCE_MODEL",
        "Rows": len(y),
        "Features": X_REFERENCE.shape[1],
        "Missing_values": int(np.isnan(X_REFERENCE).sum())
    },
    {
        "Matrix": "RR_FEATURES_ONLY_VERSION_B",
        "Rows": len(y),
        "Features": X_RR.shape[1],
        "Missing_values": int(np.isnan(X_RR).sum())
    },
    {
        "Matrix": "REFERENCE_MODEL + RR_FEATURES_VERSION_B",
        "Rows": len(y),
        "Features": X_REFERENCE_PLUS_RR.shape[1],
        "Missing_values": int(np.isnan(X_REFERENCE_PLUS_RR).sum())
    }
])

display(feature_checks)


feature_manifest = {
    "phase": "M2 — RR FEATURES — VERSION B",
    "valid_beats": EXPECTED_BEATS,
    "records": EXPECTED_RECORDS,
    "people": EXPECTED_PEOPLE,
    "sampling_rate_hz": SAMPLE_RATE_HZ,
    "reference_features": REFERENCE_FEATURE_NAMES,
    "rr_features": RR_FEATURES,
    "combined_feature_count": 17,
    "rr_policy": RR_POLICY,
    "masked_intervals": int(n_masked),
    "version": "B",
    "candidate4_explicit_mask": list(CANDIDATE4_INTERVAL),
    "post_ivr_context_intervals": [list(x) for x in sorted(KNOWN_POST_IVR_INTERVALS)],
    "rows_with_any_missing_rr": int(any_missing.sum()),
    "split_fingerprint": EXPECTED_SPLIT_FINGERPRINT,
    "splits_regenerated": False
}

(OUTPUT_DIR / "m2_feature_manifest.json").write_text(
    json.dumps(feature_manifest, indent=2)
)

print("VERSION B FEATURE CONSTRUCTION COMPLETE — PASS")
print("Training switch:", RUN_MODEL_TRAINING)

,Matrix,Rows,Features,Missing_values
0,REFERENCE_MODEL,105047,12,0
1,RR_FEATURES_ONLY_VERSION_B,105047,5,14710
2,REFERENCE_MODEL + RR_FEATURES_VERSION_B,105047,17,14710


VERSION B FEATURE CONSTRUCTION COMPLETE — PASS
Training switch: True


In [8]:
if not RUN_MODEL_TRAINING:

    print("Training skipped (RUN_MODEL_TRAINING=False). Review the checks above first.")

else:

    def make_model():
        return RandomForestClassifier(
            n_estimators=TREES,
            max_features=MAX_FEATURES,
            min_samples_split=2,
            min_samples_leaf=1,
            random_state=SEED,
            n_jobs=-1,
            class_weight=None
        )


    def to_class_order(model, raw_prob):
        out = np.zeros((raw_prob.shape[0], len(CLASS_ORDER)), dtype=np.float32)
        for j, cls in enumerate(model.classes_):
            out[:, CLASS_ORDER.index(str(cls))] = raw_prob[:, j]
        return out


    # ---- Gate: REFERENCE_MODEL must be reproducible from the features rebuilt in this notebook ----
    # Repeat 1, Fold 1 only. The paired comparison against M0's saved predictions is meaningless
    # unless the 12 reference features and Random Forest settings match M0.

    _t0 = time.time()
    assignment = fold_assignment_by_repeat[1]
    val_idx = np.flatnonzero(assignment == 1)
    train_idx = np.flatnonzero(assignment != 1)

    ref_model = make_model()
    ref_model.fit(X_REFERENCE[train_idx], y[train_idx])
    ref_pred = ref_model.predict(X_REFERENCE[val_idx])
    ref_prob = to_class_order(ref_model, ref_model.predict_proba(X_REFERENCE[val_idx]))

    saved = (
        reference_oof[(reference_oof["Repeat"] == 1) & (reference_oof["Fold"] == 1)]
        .set_index("Row_Index")
    )

    if set(saved.index) != set(row_ids[val_idx].tolist()):
        raise AssertionError("M0 reference fold 1 does not contain the same beats as the restored fold 1.")

    saved = saved.loc[row_ids[val_idx]]

    REPRO_AGREEMENT = float((ref_pred == saved["Predicted_Label"].to_numpy()).mean())
    REPRO_MAX_PROB_DIFF = float(np.abs(ref_prob - saved[PROB_COLS].to_numpy(dtype=float)).max())

    print("REFERENCE_MODEL reproduction (Repeat 1, Fold 1)")
    print("  Label agreement with M0:", round(REPRO_AGREEMENT, 6))
    print("  Max probability difference:", round(REPRO_MAX_PROB_DIFF, 6))
    print("  Seconds:", round(time.time() - _t0, 1))

    if REPRO_AGREEMENT < 0.99:
        raise AssertionError(
            "REFERENCE_MODEL features/settings do not reproduce M0 predictions. STOP. "
            "Paired comparisons would be invalid."
        )


    EXPERIMENTS = {
        "REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B": X_REFERENCE_PLUS_RR,
        "RR_FEATURES_ONLY_VERSION_B": X_RR
    }


    def run_experiment(name, X):

        pred_path = OUTPUT_DIR / f"m2_oof_predictions_{name}.csv.gz"
        audit_path = OUTPUT_DIR / f"m2_fold_run_audit_{name}.csv"

        if pred_path.exists() and audit_path.exists():
            print(f"{name}: existing complete output found, loading instead of retraining.")
            return pd.read_csv(pred_path, compression="gzip"), pd.read_csv(audit_path)

        prediction_parts = []
        audit_rows = []
        t_start = time.time()

        for repeat in range(1, N_REPEATS + 1):

            assignment = fold_assignment_by_repeat[repeat]

            for fold in range(1, N_SPLITS + 1):

                val_idx = np.flatnonzero(assignment == fold)
                train_idx = np.flatnonzero(assignment != fold)

                assert set(groups[train_idx]).isdisjoint(set(groups[val_idx]))
                assert set(record_ids[train_idx]).isdisjoint(set(record_ids[val_idx]))

                model = make_model()
                model.fit(X[train_idx], y[train_idx])

                predictions = model.predict(X[val_idx])
                probabilities = to_class_order(model, model.predict_proba(X[val_idx]))

                part = pd.DataFrame({
                    "Experiment": name,
                    "Repeat": repeat,
                    "Fold": fold,
                    "Row_Index": row_ids[val_idx],
                    "Record_ID": record_ids[val_idx],
                    "Person_ID": groups[val_idx],
                    "True_Label": y[val_idx],
                    "Predicted_Label": predictions
                })

                for j, cls in enumerate(CLASS_ORDER):
                    part[f"Prob_{cls}"] = probabilities[:, j]

                prediction_parts.append(part)

                audit_rows.append({
                    "Experiment": name,
                    "Repeat": repeat,
                    "Fold": fold,
                    "Training_Beats": len(train_idx),
                    "Validation_Beats": len(val_idx),
                    "Training_People": len(set(groups[train_idx])),
                    "Validation_People": len(set(groups[val_idx])),
                    "Training_Records": len(set(record_ids[train_idx])),
                    "Validation_Records": len(set(record_ids[val_idx]))
                })

                print(
                    f"Completed {name}: Repeat {repeat}/{N_REPEATS}, "
                    f"Fold {fold}/{N_SPLITS} "
                    f"({time.time() - t_start:.0f}s elapsed)",
                    flush=True
                )

        pred_df = pd.concat(prediction_parts, ignore_index=True)
        audit_df = pd.DataFrame(audit_rows)

        if len(pred_df) != EXPECTED_BEATS * N_REPEATS:
            raise ValueError("Incomplete OOF predictions.")

        if len(audit_df) != N_REPEATS * N_SPLITS:
            raise ValueError("Unexpected number of fold runs.")

        if pred_df.duplicated(["Repeat", "Row_Index"]).any():
            raise ValueError("Duplicate OOF row/repeat predictions.")

        if not pred_df.groupby("Repeat")["Row_Index"].nunique().eq(EXPECTED_BEATS).all():
            raise ValueError("Missing OOF rows in one or more repeats.")

        if not np.allclose(pred_df[PROB_COLS].sum(axis=1), 1.0, atol=1e-5):
            raise ValueError("Invalid probability sums.")

        pred_df.to_csv(pred_path, index=False, compression="gzip")
        audit_df.to_csv(audit_path, index=False)

        return pred_df, audit_df


    m2_predictions = {}
    m2_fold_audits = {}

    for name, X in EXPERIMENTS.items():
        m2_predictions[name], m2_fold_audits[name] = run_experiment(name, X)

    print("TRAINING COMPLETE — both experiments finished all folds.")

REFERENCE_MODEL reproduction (Repeat 1, Fold 1)
  Label agreement with M0: 1.0
  Max probability difference: 0.0
  Seconds: 65.8
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 1/5, Fold 1/5 (82s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 1/5, Fold 2/5 (166s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 1/5, Fold 3/5 (251s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 1/5, Fold 4/5 (337s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 1/5, Fold 5/5 (416s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 2/5, Fold 1/5 (506s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 2/5, Fold 2/5 (591s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 2/5, Fold 3/5 (673s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 2/5, Fold 4/5 (749s elapsed)
Completed REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B: Repeat 2

In [9]:
if not RUN_MODEL_TRAINING:

    print("Skipped because RUN_MODEL_TRAINING=False.")

else:

    def evaluate_predictions(pred, name):

        metric_rows = []
        class_rows = []
        confusion_rows = []
        s_rows = []

        for repeat in range(1, N_REPEATS + 1):

            part = pred[pred["Repeat"] == repeat]

            yt = part["True_Label"].to_numpy()
            yp = part["Predicted_Label"].to_numpy()

            precision, recall, f1, support = (
                precision_recall_fscore_support(
                    yt, yp, labels=CLASS_ORDER, zero_division=0
                )
            )

            four = part[part["True_Label"].isin(FOUR_CLASS_ORDER)]

            metric_rows.append({
                "Experiment": name,
                "Repeat": repeat,

                "Macro_F1_5_Class": f1_score(
                    yt, yp, labels=CLASS_ORDER,
                    average="macro", zero_division=0
                ),

                "Macro_F1_4_Class_N_S_V_Q": f1_score(
                    four["True_Label"], four["Predicted_Label"],
                    labels=FOUR_CLASS_ORDER,
                    average="macro", zero_division=0
                ),

                "Balanced_Accuracy": balanced_accuracy_score(yt, yp),
                "Accuracy": accuracy_score(yt, yp)
            })

            class_rows.append(pd.DataFrame({
                "Experiment": name,
                "Repeat": repeat,
                "Class": CLASS_ORDER,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "Support": support
            }))

            cm = confusion_matrix(yt, yp, labels=CLASS_ORDER)

            cm_df = pd.DataFrame(
                cm,
                index=[f"Actual_{c}" for c in CLASS_ORDER],
                columns=[f"Predicted_{c}" for c in CLASS_ORDER]
            )

            cm_df.insert(0, "Actual_Class", cm_df.index)
            cm_df["Repeat"] = repeat
            cm_df["Experiment"] = name

            confusion_rows.append(cm_df.reset_index(drop=True))

            # S probability diagnostic.
            mean_s_on_s = part.loc[part["True_Label"] == "S", "Prob_S"].mean()
            mean_s_on_n = part.loc[part["True_Label"] == "N", "Prob_S"].mean()

            s_rows.append({
                "Experiment": name,
                "Repeat": repeat,
                "Mean_Prob_S_on_true_S": mean_s_on_s,
                "Mean_Prob_S_on_true_N": mean_s_on_n,
                "Difference_S_minus_N": mean_s_on_s - mean_s_on_n
            })

        repeat_metrics = pd.DataFrame(metric_rows)
        per_class = pd.concat(class_rows, ignore_index=True)
        confusions = pd.concat(confusion_rows, ignore_index=True)
        s_diagnostic = pd.DataFrame(s_rows)

        headline = {
            "Experiment": name,
            "Headline_Macro_F1_5_Class": repeat_metrics["Macro_F1_5_Class"].mean(),
            "Macro_F1_4_Class_N_S_V_Q": repeat_metrics["Macro_F1_4_Class_N_S_V_Q"].mean(),
            "Balanced_Accuracy": repeat_metrics["Balanced_Accuracy"].mean(),
            "Accuracy": repeat_metrics["Accuracy"].mean()
        }

        return headline, repeat_metrics, per_class, confusions, s_diagnostic


    # The saved M0 reference predictions are evaluated the same way, as the comparison row.
    all_to_evaluate = {"REFERENCE_MODEL": reference_oof}
    all_to_evaluate.update(m2_predictions)

    evaluations = {}
    headline_rows = []

    for name, pred in all_to_evaluate.items():

        ev = evaluate_predictions(pred, name)

        evaluations[name] = ev
        headline_rows.append(ev[0])

        print("\n", name, "— metrics by repeat")
        display(ev[1])

        print("Per-class metrics")
        display(ev[2])

        print("Confusion matrices")
        display(ev[3])

        print("S probability diagnostic")
        display(ev[4])

        ev[1].to_csv(OUTPUT_DIR / f"m2_repeat_metrics_{name}.csv", index=False)
        ev[2].to_csv(OUTPUT_DIR / f"m2_per_class_metrics_{name}.csv", index=False)
        ev[3].to_csv(OUTPUT_DIR / f"m2_confusion_matrices_{name}.csv", index=False)
        ev[4].to_csv(OUTPUT_DIR / f"m2_s_probability_diagnostic_{name}.csv", index=False)


    m2_headline = pd.DataFrame(headline_rows)

    print("\nM2 HEADLINE RESULTS (REFERENCE_MODEL row = saved M0 predictions)")
    display(m2_headline)

    m2_headline.to_csv(OUTPUT_DIR / "m2_headline_results.csv", index=False)


 REFERENCE_MODEL — metrics by repeat


,Experiment,Repeat,Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,1,0.518445,0.651288,0.535339,0.913344
1,REFERENCE_MODEL,2,0.316230,0.397515,0.364539,0.889211
2,REFERENCE_MODEL,3,0.520605,0.653948,0.540737,0.912877
3,REFERENCE_MODEL,4,0.312341,0.392648,0.362271,0.875903
4,REFERENCE_MODEL,5,0.508729,0.638717,0.530906,0.901501


Per-class metrics


,Experiment,Repeat,Class,Precision,Recall,F1,Support
0,REFERENCE_MODEL,1,N,0.948918,0.956365,0.952627,90341
1,REFERENCE_MODEL,1,S,0.015385,0.000360,0.000703,2781
2,REFERENCE_MODEL,1,V,0.610891,0.853507,0.712101,7229
3,REFERENCE_MODEL,1,F,0.000000,0.000000,0.000000,802
4,REFERENCE_MODEL,1,Q,0.996162,0.866461,0.926796,3894
5,REFERENCE_MODEL,2,N,0.949499,0.965453,0.957409,90341
6,REFERENCE_MODEL,2,S,0.000000,0.000000,0.000000,2781
7,REFERENCE_MODEL,2,V,0.488591,0.855997,0.622097,7229
8,REFERENCE_MODEL,2,F,0.002404,0.001247,0.001642,802
9,REFERENCE_MODEL,2,Q,0.000000,0.000000,0.000000,3894


Confusion matrices


,Actual_Class,Predicted_N,Predicted_S,Predicted_V,Predicted_F,Predicted_Q,Repeat,Experiment
0,Actual_N,86399,63,3469,398,12,1,REFERENCE_MODEL
1,Actual_S,2629,1,139,12,0,1,REFERENCE_MODEL
2,Actual_V,1022,1,6170,35,1,1,REFERENCE_MODEL
3,Actual_F,563,0,239,0,0,1,REFERENCE_MODEL
4,Actual_Q,437,0,83,0,3374,1,REFERENCE_MODEL
5,Actual_N,87220,96,2661,360,4,2,REFERENCE_MODEL
6,Actual_S,2641,0,133,7,0,2,REFERENCE_MODEL
7,Actual_V,986,6,6188,48,1,2,REFERENCE_MODEL
8,Actual_F,561,0,240,1,0,2,REFERENCE_MODEL
9,Actual_Q,451,0,3443,0,0,2,REFERENCE_MODEL


... (16 outputs left out)

S probability diagnostic


,Experiment,Repeat,Mean_Prob_S_on_true_S,Mean_Prob_S_on_true_N,Difference_S_minus_N
0,RR_FEATURES_ONLY_VERSION_B,1,0.078025,0.012951,0.065074
1,RR_FEATURES_ONLY_VERSION_B,2,0.086995,0.012589,0.074406
2,RR_FEATURES_ONLY_VERSION_B,3,0.113452,0.010783,0.102668
3,RR_FEATURES_ONLY_VERSION_B,4,0.073727,0.012713,0.061013
4,RR_FEATURES_ONLY_VERSION_B,5,0.090888,0.011451,0.079437



M2 HEADLINE RESULTS (REFERENCE_MODEL row = saved M0 predictions)


,Experiment,Headline_Macro_F1_5_Class,Macro_F1_4_Class_N_S_V_Q,Balanced_Accuracy,Accuracy
0,REFERENCE_MODEL,0.435270,0.546823,0.466758,0.898567
1,REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B,0.499921,0.623839,0.503265,0.911146
2,RR_FEATURES_ONLY_VERSION_B,0.333523,0.417145,0.323652,0.866071


In [10]:
if not RUN_MODEL_TRAINING:

    print("Skipped because RUN_MODEL_TRAINING=False.")

else:

    # Record-level bootstrap computed from per-(repeat, record) confusion counts.
    # Same resampling scheme as the original loop (same seed, same records drawn for
    # candidate and baseline), but each draw just sums small 5x5 count matrices, so the
    # whole cell finishes in seconds instead of about an hour.

    METRIC_CLASSES = {
        "Macro_F1_5_Class": CLASS_ORDER,
        "Macro_F1_4_Class_N_S_V_Q": FOUR_CLASS_ORDER
    }

    CLASS_INDEX = {c: i for i, c in enumerate(CLASS_ORDER)}


    def macro_f1_from_counts(M, scored_classes):
        # M: 5x5 counts, rows = true, columns = predicted (CLASS_ORDER).
        # Rows are restricted to the scored classes; predictions outside them count as errors
        # but are not scored as classes. Matches sklearn f1_score(labels=..., zero_division=0).
        ids = [CLASS_INDEX[c] for c in scored_classes]
        rows = M[ids, :]
        f1s = []
        for c in ids:
            tp = M[c, c]
            fn = M[c, :].sum() - tp
            fp = rows[:, c].sum() - tp
            denom = 2 * tp + fp + fn
            f1s.append(0.0 if denom == 0 else 2.0 * tp / denom)
        return float(np.mean(f1s))


    def build_blocks(df, true_col, pred_col, record_col):
        t = df[true_col].map(CLASS_INDEX).to_numpy()
        p = df[pred_col].map(CLASS_INDEX).to_numpy()
        if np.isnan(t.astype(float)).any() or np.isnan(p.astype(float)).any():
            raise ValueError("Unknown class label in predictions.")
        rep = df["Repeat"].to_numpy()
        rec = df[record_col].to_numpy()
        flat = t * len(CLASS_ORDER) + p
        blocks = {}
        for r in np.unique(rep):
            in_r = rep == r
            for c in np.unique(rec[in_r]):
                sel = in_r & (rec == c)
                blocks[(int(r), int(c))] = np.bincount(
                    flat[sel], minlength=len(CLASS_ORDER) ** 2
                ).reshape(len(CLASS_ORDER), len(CLASS_ORDER)).astype(np.int64)
        return blocks


    def point_score(blocks, metric):
        repeats = sorted({k[0] for k in blocks})
        scores = []
        for r in repeats:
            M = sum(v for (rr, _), v in blocks.items() if rr == r)
            scores.append(macro_f1_from_counts(M, METRIC_CLASSES[metric]))
        return float(np.mean(scores))


    def bootstrap_scores(blocks_list, metric, records, repeats):
        rng = np.random.default_rng(BOOTSTRAP_SEED)
        out = [[] for _ in blocks_list]
        for _ in range(BOOTSTRAP_SAMPLES):
            sampled = rng.choice(records, size=len(records), replace=True)
            for k, blocks in enumerate(blocks_list):
                per_repeat = []
                for rep in repeats:
                    M = sum(blocks[(int(rep), int(rec))] for rec in sampled)
                    per_repeat.append(macro_f1_from_counts(M, METRIC_CLASSES[metric]))
                out[k].append(float(np.mean(per_repeat)))
        return [np.asarray(x) for x in out]


    def record_ci(pred, metric):

        blocks = build_blocks(pred, "True_Label", "Predicted_Label", "Record_ID")
        records = np.array(sorted(pred["Record_ID"].unique()))
        repeats = sorted(pred["Repeat"].unique())

        (scores,) = bootstrap_scores([blocks], metric, records, repeats)
        lower, upper = np.quantile(scores, [0.025, 0.975])

        return {
            "Metric": metric,
            "Point_Estimate": point_score(blocks, metric),
            "CI_95_Lower": float(lower),
            "CI_95_Upper": float(upper),
            "Bootstrap_Samples": BOOTSTRAP_SAMPLES,
            "Resampling_Unit": "Record_ID",
            "Bootstrap_Seed": BOOTSTRAP_SEED
        }


    def paired_record_bootstrap(candidate, baseline, metric):

        keys = ["Repeat", "Row_Index"]
        meta = ["Record_ID", "Person_ID", "True_Label"]

        merged = candidate.merge(
            baseline[keys + meta + ["Predicted_Label"]],
            on=keys,
            how="inner",
            suffixes=("_candidate", "_baseline"),
            validate="one_to_one"
        )

        if len(merged) != EXPECTED_BEATS * N_REPEATS:
            raise ValueError("Candidate and reference OOF predictions do not align.")

        for col in meta:
            if not (merged[f"{col}_candidate"].astype(str) == merged[f"{col}_baseline"].astype(str)).all():
                raise ValueError(f"Candidate/reference mismatch in {col}.")

        cand = pd.DataFrame({
            "Repeat": merged["Repeat"],
            "Record_ID": merged["Record_ID_candidate"],
            "True_Label": merged["True_Label_candidate"],
            "Predicted_Label": merged["Predicted_Label_candidate"]
        })
        base = pd.DataFrame({
            "Repeat": merged["Repeat"],
            "Record_ID": merged["Record_ID_candidate"],
            "True_Label": merged["True_Label_candidate"],
            "Predicted_Label": merged["Predicted_Label_baseline"]
        })

        bc = build_blocks(cand, "True_Label", "Predicted_Label", "Record_ID")
        bb = build_blocks(base, "True_Label", "Predicted_Label", "Record_ID")

        records = np.array(sorted(cand["Record_ID"].unique()))
        repeats = sorted(cand["Repeat"].unique())

        point = point_score(bc, metric) - point_score(bb, metric)

        sc, sb = bootstrap_scores([bc, bb], metric, records, repeats)
        diff = sc - sb
        lower, upper = np.quantile(diff, [0.025, 0.975])

        return {
            "Metric": metric,
            "Candidate_minus_REFERENCE_MODEL": float(point),
            "Paired_CI_95_Lower": float(lower),
            "Paired_CI_95_Upper": float(upper),
            "Bootstrap_Samples": BOOTSTRAP_SAMPLES,
            "Resampling_Unit": "Record_ID",
            "Bootstrap_Seed": BOOTSTRAP_SEED,
            "Retention_Rule": "KEEP only if paired CI lower bound > 0",
            "Decision": (
                "KEEP"
                if lower > 0
                else "NO MEASURABLE EFFECT — retain simpler reference"
            )
        }


    ci_rows = []
    paired_rows = []

    for name, candidate in m2_predictions.items():

        for metric in METRIC_CLASSES:

            ci = record_ci(candidate, metric)
            ci["Experiment"] = name
            ci_rows.append(ci)

            paired = paired_record_bootstrap(candidate, reference_oof, metric)
            paired["Experiment"] = name

            if name == "RR_FEATURES_ONLY_VERSION_B":
                # Diagnostic run: it answers "how much signal do RR features carry alone?"
                # It is not a candidate replacement for REFERENCE_MODEL.
                paired["Retention_Rule"] = "Not applicable (diagnostic)"
                paired["Decision"] = "DIAGNOSTIC ONLY — not a retention decision"

            paired_rows.append(paired)

    # Reference row for context.
    for metric in METRIC_CLASSES:
        ci = record_ci(reference_oof, metric)
        ci["Experiment"] = "REFERENCE_MODEL"
        ci_rows.append(ci)


    m2_ci = pd.DataFrame(ci_rows)
    paired_results = pd.DataFrame(paired_rows)

    print("M2 RECORD-LEVEL CONFIDENCE INTERVALS")
    display(m2_ci)

    print("\nPAIRED IMPROVEMENT VS REFERENCE_MODEL")
    display(paired_results)

    m2_ci.to_csv(OUTPUT_DIR / "m2_record_bootstrap_ci.csv", index=False)

    paired_results.to_csv(
        OUTPUT_DIR / "m2_paired_record_bootstrap_vs_REFERENCE_MODEL.csv",
        index=False
    )

M2 RECORD-LEVEL CONFIDENCE INTERVALS


,Metric,Point_Estimate,CI_95_Lower,CI_95_Upper,Bootstrap_Samples,Resampling_Unit,Bootstrap_Seed,Experiment
0,Macro_F1_5_Class,0.499921,0.374965,0.554566,1000,Record_ID,20261105,REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B
1,Macro_F1_4_Class_N_S_V_Q,0.623839,0.466300,0.695583,1000,Record_ID,20261105,REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B
2,Macro_F1_5_Class,0.333523,0.305838,0.357070,1000,Record_ID,20261105,RR_FEATURES_ONLY_VERSION_B
3,Macro_F1_4_Class_N_S_V_Q,0.417145,0.383110,0.446281,1000,Record_ID,20261105,RR_FEATURES_ONLY_VERSION_B
4,Macro_F1_5_Class,0.435270,0.311816,0.467416,1000,Record_ID,20261105,REFERENCE_MODEL
5,Macro_F1_4_Class_N_S_V_Q,0.546823,0.393360,0.588774,1000,Record_ID,20261105,REFERENCE_MODEL



PAIRED IMPROVEMENT VS REFERENCE_MODEL


,Metric,Candidate_minus_REFERENCE_MODEL,Paired_CI_95_Lower,Paired_CI_95_Upper,Bootstrap_Samples,Resampling_Unit,Bootstrap_Seed,Retention_Rule,Decision,Experiment
0,Macro_F1_5_Class,0.064651,0.027660,0.115754,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,KEEP,REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B
1,Macro_F1_4_Class_N_S_V_Q,0.077016,0.032285,0.142478,1000,Record_ID,20261105,KEEP only if paired CI lower bound > 0,KEEP,REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B
2,Macro_F1_5_Class,-0.101748,-0.138055,0.015480,1000,Record_ID,20261105,Not applicable (diagnostic),DIAGNOSTIC ONLY — not a retention decision,RR_FEATURES_ONLY_VERSION_B
3,Macro_F1_4_Class_N_S_V_Q,-0.129678,-0.176188,0.017175,1000,Record_ID,20261105,Not applicable (diagnostic),DIAGNOSTIC ONLY — not a retention decision,RR_FEATURES_ONLY_VERSION_B


In [11]:
if not RUN_MODEL_TRAINING:

    print("Skipped because RUN_MODEL_TRAINING=False.")

else:

    checks = {

        "Corrected_dataset_selected":
            corrected_path.name == "MITBIH_MLII_DualLead_Dataset_corrected.csv",

        "M1_positions_loaded":
            bool(aami_df["Sample_Index"].notna().all()),

        "M1_waveforms_match_corrected_dataset":
            bool(np.array_equal(processed, reconstructed)),

        "Annotation_beats_match_M1_positions_and_symbols":
            bool(ANNOTATION_BEATS_MATCH_M1),

        "105047_valid_beats":
            len(aami_df) == EXPECTED_BEATS,

        "Version_B_RR_rows_preserved":
            bool(RR_FEATURE_ROWS_PRESERVED),

        "Version_B_composite_keys_unique":
            bool(RR_FEATURE_KEYS_UNIQUE),

        "Version_B_sample_order_valid_all_records":
            bool(RR_SAMPLE_ORDER_VALID) and len(sample_order_check) == EXPECTED_RECORDS,

        "Expected_class_counts":
            counts == EXPECTED_COUNTS,

        "46_records_45_people":
            aami_df.Record_ID.nunique() == EXPECTED_RECORDS
            and aami_df.Person_ID.nunique() == EXPECTED_PEOPLE,

        "Original_split_fingerprint_verified":
            computed_fp == EXPECTED_SPLIT_FINGERPRINT,

        "No_new_splits_generated":
            True,

        "RR_no_infinite_values":
            bool(not np.isinf(X_RR).any()),

        "RR_missing_counts_match_policy":
            bool(RR_NAN_COUNTS_MATCH_POLICY),

        "Version_B_strict_previous_10_of_10":
            RR_WINDOW == 10 and RR_MIN_VALID == 10,

        "Version_B_marker_mask_count_687":
            n_marker_masked == EXPECTED_VERSION_A_MARKER_MASKED_INTERVALS,

        "Version_B_total_mask_count_688":
            n_masked == EXPECTED_VERSION_B_MASKED_INTERVALS,

        "Candidate4_explicitly_masked_once":
            int(interval_qc["Candidate4_Explicit_QC_Mask"].sum()) == 1
            and int((interval_qc["Is_Masked"] & ~interval_qc["Marker_Masked"]).sum()) == 1,

        "Two_post_IVR_E_to_E_intervals_retained":
            len(post_ivr_rows) == 2 and not post_ivr_rows["Is_Masked"].any(),

        "Raw_pipe_markers_all_masked_when_inside_intervals":
            len(pipe_unmasked_intervals) == 0,

        "No_unmasked_interval_over_60s":
            bool(NO_UNMASKED_INTERVAL_OVER_60S),

        "17_feature_combination":
            X_REFERENCE_PLUS_RR.shape == (EXPECTED_BEATS, 17),

        "REFERENCE_MODEL_reproduced_from_M0":
            REPRO_AGREEMENT >= 0.99,

        "Both_experiments_all_folds":
            all(len(a) == N_REPEATS * N_SPLITS for a in m2_fold_audits.values()),

        "Both_OOF_outputs_complete":
            all(
                len(p) == EXPECTED_BEATS * N_REPEATS
                for p in m2_predictions.values()
            ),

        "Reference_OOF_aligned":
            len(reference_oof) == EXPECTED_BEATS * N_REPEATS
    }


    check_table = pd.DataFrame([
        {"Check": name, "Result": "PASS" if passed else "FAIL"}
        for name, passed in checks.items()
    ])

    display(check_table)

    check_table.to_csv(OUTPUT_DIR / "m2_final_validation_checks.csv", index=False)

    if not all(checks.values()):
        raise AssertionError("M2 final validation failed. STOP.")


    manifest = {

        "phase": "M2 — RR FEATURES — VERSION B",

        "experiments": [
            "REFERENCE_MODEL_PLUS_RR_FEATURES_VERSION_B",
            "RR_FEATURES_ONLY_VERSION_B"
        ],

        "corrected_dataset": str(corrected_path),
        "m1_verified_beat_table": str(beat_table_path),
        "m0_reference_oof": str(reference_oof_path),

        "valid_beats": EXPECTED_BEATS,
        "records": EXPECTED_RECORDS,
        "people": EXPECTED_PEOPLE,
        "class_counts": counts,
        "sampling_rate_hz": SAMPLE_RATE_HZ,

        "rr_feature_names": RR_FEATURES,
        "rr_policy": RR_POLICY,
        "masked_intervals": int(n_masked),
        "marker_masked_intervals": int(n_marker_masked),
        "candidate4_interval": list(CANDIDATE4_INTERVAL),
        "post_ivr_context_intervals": [list(x) for x in sorted(KNOWN_POST_IVR_INTERVALS)],
        "raw_pipe_marker_count": int(raw_pipe_marker_count),
        "pipe_markers_inside_intervals": int(pipe_markers_inside_count),
        "intervals_with_pipe_markers": int(len(pipe_intervals)),
        "rows_with_any_missing_rr": int(any_missing.sum()),

        "reference_reproduction_label_agreement": REPRO_AGREEMENT,
        "reference_reproduction_max_probability_difference": REPRO_MAX_PROB_DIFF,

        "split_fingerprint": computed_fp,
        "splits_regenerated": False,

        "random_forest": {
            "n_estimators": TREES,
            "max_features": MAX_FEATURES,
            "random_state": SEED,
            "class_weight": None,
            "n_jobs": -1
        },

        "bootstrap_samples": BOOTSTRAP_SAMPLES,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "bootstrap_unit": "Record_ID",

        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,

        "headline_results": m2_headline.to_dict(orient="records"),
        "paired_results": paired_results.to_dict(orient="records"),

        "checks": checks
    }

    (OUTPUT_DIR / "m2_reproducibility_manifest.json").write_text(
        json.dumps(manifest, indent=2)
    )

    print("M2 VERSION B COMPLETE.")
    print("Results saved to:", OUTPUT_DIR)
    print("Do not start M3 until M2 results are reviewed.")

,Check,Result
0,Corrected_dataset_selected,PASS
1,M1_positions_loaded,PASS
2,M1_waveforms_match_corrected_dataset,PASS
3,Annotation_beats_match_M1_positions_and_symbols,PASS
4,105047_valid_beats,PASS
5,Version_B_RR_rows_preserved,PASS
6,Version_B_composite_keys_unique,PASS
7,Version_B_sample_order_valid_all_records,PASS
8,Expected_class_counts,PASS
9,46_records_45_people,PASS


M2 VERSION B COMPLETE.
Results saved to: /kaggle/working/ECG_Project/05_M2_RR_FEATURES_VERSION_B
Do not start M3 until M2 results are reviewed.
